In [1]:
import qlib
import pandas as pd
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict

In [2]:
# use default data
# NOTE: need to download data from remote: python scripts/get_data.py qlib_data_cn --target_dir ~/.qlib/qlib_data/cn_data
provider_uri = "~/.qlib/qlib_data/cn_data"  # target_dir
if not exists_qlib_data(provider_uri):
    print(f"Qlib data is not found in {provider_uri}")
    sys.path.append(str(scripts_dir))
    from get_data import GetData

    GetData().qlib_data(target_dir=provider_uri, region=REG_CN)
qlib.init(provider_uri=provider_uri, region=REG_CN)

[22642:MainThread](2025-03-10 07:35:43,898) INFO - qlib.Initialization - [config.py:420] - default_conf: client.
[22642:MainThread](2025-03-10 07:35:43,901) INFO - qlib.Initialization - [__init__.py:74] - qlib successfully initialized based on client settings.
[22642:MainThread](2025-03-10 07:35:43,902) INFO - qlib.Initialization - [__init__.py:76] - data_path={'__DEFAULT_FREQ': PosixPath('/home/hlst/.qlib/qlib_data/cn_data')}


In [3]:
market = "csi300"
benchmark = "SH000300"

In [17]:
data_handler_config = {
    "start_time": "2008-01-01",
    "end_time": "2025-02-28",
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2014-12-31",
    "instruments": market,
}

dataset_config = {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": ("2008-01-01", "2014-12-31"),
                "valid": ("2015-01-01", "2016-12-31"),
                "test": ("2017-01-01", "2020-08-01"),
            },
        },
    }

dataset = init_instance_by_config(dataset_config)

[22642:MainThread](2025-03-10 07:47:01,429) INFO - qlib.timer - [log.py:127] - Time cost: 14.557s | Loading data Done
[22642:MainThread](2025-03-10 07:47:02,027) INFO - qlib.timer - [log.py:127] - Time cost: 0.128s | DropnaLabel Done
[22642:MainThread](2025-03-10 07:47:03,775) INFO - qlib.timer - [log.py:127] - Time cost: 1.747s | CSZScoreNorm Done
[22642:MainThread](2025-03-10 07:47:03,777) INFO - qlib.timer - [log.py:127] - Time cost: 2.346s | fit & process data Done
[22642:MainThread](2025-03-10 07:47:03,778) INFO - qlib.timer - [log.py:127] - Time cost: 16.907s | Init data Done


In [14]:
rid = "e148f994ae0b41ae830ff7e5b687ad33"

In [24]:
###################################
# prediction, backtest & analysis
###################################

# backtest and analysis
with R.start(experiment_name="backtest_analysis"):
    recorder = R.get_recorder(recorder_id=rid, experiment_name="train_model")
    model = recorder.load_object("trained_model")

    # prediction
    recorder = R.get_recorder()
    ba_rid = recorder.id
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()

    port_analysis_config = {
        "executor": {
            "class": "SimulatorExecutor",
            "module_path": "qlib.backtest.executor",
            "kwargs": {
                "time_per_step": "day",
                "generate_portfolio_metrics": True,
            },
        },
        "strategy": {
            "class": "TopkDropoutStrategy",
            "module_path": "qlib.contrib.strategy.signal_strategy",
            "kwargs": {
                "model": model,
                "dataset": dataset,
                "topk": 50,
                "n_drop": 5,
            },
        },
        "backtest": {
            "start_time": "2017-01-01",
            "end_time": "2020-08-01",
            "account": 100000000,
            "benchmark": benchmark,
            "exchange_kwargs": {
                "freq": "day",
                "limit_threshold": 0.095,
                "deal_price": "close",
                "open_cost": 0.0005,
                "close_cost": 0.0015,
                "min_cost": 5,
            },
        },
    }

    # backtest & analysis
    par = PortAnaRecord(recorder, port_analysis_config, "day")
    par.generate()

[22642:MainThread](2025-03-10 08:02:13,513) INFO - qlib.workflow - [exp.py:258] - Experiment 208409257753392152 starts running ...
[22642:MainThread](2025-03-10 08:02:13,520) INFO - qlib.workflow - [recorder.py:345] - Recorder 07540a8cccd54541acf51d835a2b3913 starts running under Experiment 208409257753392152 ...
[22642:MainThread](2025-03-10 08:02:14,769) INFO - qlib.workflow - [record_temp.py:198] - Signal record 'pred.pkl' has been saved as the artifact of the Experiment 208409257753392152
[22642:MainThread](2025-03-10 08:02:14,786) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange


'The following are prediction results of the LGBModel model.'
                          score
datetime   instrument          
2017-01-03 SH600000   -0.046878
           SH600005   -0.096360
           SH600008    0.000686
           SH600009    0.027598
           SH600010   -0.023481


[22642:MainThread](2025-03-10 08:02:18,400) WARNING - qlib.online operator - [exchange.py:219] - $close field data contains nan.
[22642:MainThread](2025-03-10 08:02:18,403) WARNING - qlib.online operator - [exchange.py:219] - $close field data contains nan.
[22642:MainThread](2025-03-10 08:02:32,062) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x7fe7abe8c3a0>


backtest loop:   0%|          | 0/871 [00:00<?, ?it/s]

/home/hlst/.pyenv/versions/3.10.12/lib/python3.10/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
[22642:MainThread](2025-03-10 08:02:39,155) INFO - qlib.workflow - [record_temp.py:515] - Portfolio analysis record 'port_analysis_1day.pkl' has been saved as the artifact of the Experiment 208409257753392152
[22642:MainThread](2025-03-10 08:02:39,162) INFO - qlib.workflow - [record_temp.py:540] - Indicator analysis record 'indicator_analysis_1day.pkl' has been saved as the artifact of the Experiment 208409257753392152


'The following are analysis results of benchmark return(1day).'
                       risk
mean               0.000477
std                0.012295
annualized_return  0.113561
information_ratio  0.598699
max_drawdown      -0.370479
'The following are analysis results of the excess return without cost(1day).'
                       risk
mean               0.000619
std                0.005625
annualized_return  0.147363
information_ratio  1.698290
max_drawdown      -0.085673
'The following are analysis results of the excess return with cost(1day).'
                       risk
mean               0.000465
std                0.005623
annualized_return  0.110718
information_ratio  1.276433
max_drawdown      -0.088253
'The following are analysis results of indicators(1day).'
     value
ffr    1.0
pa     0.0
pos    0.0


[22642:MainThread](2025-03-10 08:02:39,711) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


In [26]:
par.recorder.list_artifacts()

['code_cached.txt',
 'code_diff.txt',
 'code_status.txt',
 'label.pkl',
 'portfolio_analysis',
 'pred.pkl']

In [32]:
a = sr.load("pred.pkl")

In [41]:
a.dtypes, a.index

(score    float64
 dtype: object,
 MultiIndex([('2017-01-03', 'SH600000'),
             ('2017-01-03', 'SH600005'),
             ('2017-01-03', 'SH600008'),
             ('2017-01-03', 'SH600009'),
             ('2017-01-03', 'SH600010'),
             ('2017-01-03', 'SH600015'),
             ('2017-01-03', 'SH600016'),
             ('2017-01-03', 'SH600018'),
             ('2017-01-03', 'SH600019'),
             ('2017-01-03', 'SH600021'),
             ...
             ('2020-07-31', 'SZ300136'),
             ('2020-07-31', 'SZ300142'),
             ('2020-07-31', 'SZ300144'),
             ('2020-07-31', 'SZ300347'),
             ('2020-07-31', 'SZ300408'),
             ('2020-07-31', 'SZ300413'),
             ('2020-07-31', 'SZ300433'),
             ('2020-07-31', 'SZ300498'),
             ('2020-07-31', 'SZ300601'),
             ('2020-07-31', 'SZ300628')],
            names=['datetime', 'instrument'], length=261207))

In [42]:
a_last = a.loc['2020-07-31']

In [43]:
a_last

,score
instrument,
SH600000,-0.042360
SH600004,-0.044743
SH600009,-0.040727
SH600010,-0.066453
SH600011,-0.068240
...,...
SZ300413,-0.036455
SZ300433,-0.090913
SZ300498,-0.031189


In [44]:
a_last.sort_values(by='score')

,score
instrument,
SH603259,-0.297016
SH603899,-0.285293
SZ300014,-0.266055
SH600809,-0.260123
SZ300601,-0.254045
...,...
SZ000786,0.130098
SH600989,0.158752
SH601877,0.163728
